Purpose: Look at DE gene results from maSigPro (parents & 3 Yg phys categories).<br>
Author: Anna Pardo<br>
Date initiated: Apr. 21, 2026

In [1]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

In [2]:
# set directory paths
local = "./DE_results/"
hpc = "./DEGs_results/"

In [56]:
# load parental DEGs - Yf
yfde = pd.read_csv(os.path.join(local,"degs_Yf.txt"),delim_whitespace=True,quotechar='"')
yfde.head()

,X118,X102,X4,X10,X149,X66,X67,X16,X101,X132,...,X104,X70,X105,X131,X114,X130,X29,X150,X35,X72
YufilH1000134m.g,25.988774,20.813978,23.293301,23.354448,20.750916,24.739007,20.511402,22.124096,22.870749,22.736530,...,22.359780,24.096139,16.465059,23.711373,18.598261,20.683284,19.192685,16.671503,20.274475,16.929788
YufilH1000358m.g,34.244032,24.613355,34.062063,38.514353,24.870583,21.778784,33.840360,38.214347,23.457179,27.999616,...,22.359780,31.105925,34.481175,23.407381,45.390804,29.820762,29.773268,23.729512,25.710386,25.193136
YufilH1000415m.g,12.382887,14.371557,14.280315,9.013998,10.756909,16.563153,23.135756,12.642340,8.063405,19.859376,...,3.855135,37.589977,24.458965,14.794276,31.764356,34.920750,12.795123,22.390924,13.516317,23.903248
YufilH1000476m.g,1.528751,1.982284,0.936414,2.185212,4.958859,5.215631,5.110585,4.309889,2.492325,8.070065,...,4.780367,3.680138,6.741129,6.485162,7.089436,5.737486,6.397562,4.380833,2.644497,5.119245
YufilH1000612m.g,21.249645,17.344982,22.239835,23.627600,23.115910,26.782971,25.898235,19.250837,20.085209,34.034620,...,19.121467,26.461942,24.339653,31.615164,28.265673,32.087423,27.435697,39.914256,30.558630,35.754099


In [57]:
# really I just want the DEG list out of this
yfde = list(yfde.index)

In [58]:
len(yfde)

794

In [13]:
def get_degs_file(filepath):
    df = pd.read_csv(filepath,delim_whitespace=True,quotechar='"')
    genes = list(df.index)
    return genes

In [19]:
# get Ya DEGs
yade = get_degs_file(os.path.join(local,"degs_Ya.txt"))
len(yade)

809

In [18]:
# get Yg DEGs run locally (three subsets of C3+CAM Yg)
localruns = {}
for f in os.listdir(local):
    if (not f.endswith("_pval.txt")) and (not f.endswith("_R2.txt")):
        if ("Yf" not in f) and ("Ya" not in f):
            ID = f.split("s_")[1].split(".")[0]
            localruns[ID] = get_degs_file(os.path.join(local,f))

In [23]:
# get the rest of the information
hpcruns = {}
for f in os.listdir(hpc):
    if f.endswith("genes.txt"):
        ID = f.split("_3reps")[0]
        hpcruns[ID] = get_degs_file(os.path.join(hpc,f))

In [26]:
localruns.update(hpcruns)

In [32]:
# localruns now contains all the DEG info
## split into three dicts: one for each physiology
camd = {k:v for k,v in localruns.items() if ("facultative" not in k) and ("C3" not in k)}

In [34]:
fcd = {k:v for k,v in localruns.items() if "facultative" in k}

In [35]:
c3d = {k:v for k,v in localruns.items() if "C3" in k}

In [44]:
from collections import Counter

def genes_in_at_least_x_entries(d, max_x=10):
    """
    d: dictionary of lists {key: [genes, ...]}
    max_x: highest X to report
    
    Returns:
        counts_by_x -> {X: number of genes found in at least X entries}
        gene_freqs   -> Counter of how many entries each gene appears in
    """
    
    # Count how many dictionary entries each gene appears in
    gene_freqs = Counter()
    
    for gene_list in d.values():
        for gene in set(gene_list):   # avoids duplicates within one list
            gene_freqs[gene] += 1

    # Count genes present in at least X entries
    counts_by_x = {
        x: sum(freq >= x for freq in gene_freqs.values())
        for x in range(1, max_x + 1)
    }

    return gene_freqs

In [40]:
genes_in_at_least_x_entries(fcd)

{1: 5926, 2: 1726, 3: 766, 4: 362, 5: 181, 6: 90, 7: 37, 8: 14, 9: 3, 10: 1}

In [41]:
genes_in_at_least_x_entries(camd)

{1: 12106,
 2: 5428,
 3: 2908,
 4: 1575,
 5: 885,
 6: 472,
 7: 226,
 8: 112,
 9: 40,
 10: 8}

In [42]:
genes_in_at_least_x_entries(c3d)

{1: 10113,
 2: 3969,
 3: 2076,
 4: 1154,
 5: 643,
 6: 348,
 7: 171,
 8: 80,
 9: 28,
 10: 7}

In [45]:
camfreqs = genes_in_at_least_x_entries(camd)
c3freqs = genes_in_at_least_x_entries(c3d)
facfreqs = genes_in_at_least_x_entries(fcd)

In [46]:
camgenes_by_x = {
    x: [gene for gene, freq in camfreqs.items() if freq >= x]
    for x in range(1, 11)
}

In [47]:
c3cam_by_x = {
    x: [gene for gene, freq in c3freqs.items() if freq >= x]
    for x in range(1, 11)
}

In [48]:
faccam_by_x = {
    x: [gene for gene, freq in facfreqs.items() if freq >= x]
    for x in range(1, 11)
}

In [51]:
# save each of these as a JSON
with open("./CAM_DEGs_by_x_overlap.json","w+") as outfile:
    json.dump(camgenes_by_x,outfile)

In [52]:
with open("./facCAM_DEGs_by_x_overlap.json","w+") as outfile:
    json.dump(faccam_by_x,outfile)

In [53]:
with open("./C3+CAM_DEGs_by_x_overlap.json","w+") as outfile:
    json.dump(c3cam_by_x,outfile)

In [54]:
# save parental DEG lists
with open("./Ya_DEGs_masigpro.txt","w+") as outfile:
    for i in yade:
        outfile.write(i+"\n")

In [59]:
with open("./Yf_DEGs_masigpro.txt","w+") as outfile:
    for i in yfde:
        outfile.write(i+"\n")